# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline scoring rule (plain English):**

A page's refresh urgency score is the weighted sum of five normalized risk signals:

1. **Position risk (25%):** Higher avg_position = worse rank = more urgent. Normalized 0–1 within the dataset.
2. **CTR decay (20%):** Lower CTR at a given position = losing the SERP competition.
3. **Staleness (25%):** More days since last update = staler content = more urgent.
4. **Volume drop signal (20%):** Ratio of recent impressions to prior impressions — lower = more decline signal.
5. **Engagement deficit (10%):** Lower engagement_rate = visitors are not finding the content useful.

Pages with missing data in any signal get the population median for that signal.

**Reason codes:**
- `HIGH_POSITION` — avg_position > 20 (falling off page 1)
- `LOW_CTR` — ctr below 25th percentile for its position tier
- `STALE_CONTENT` — days_since_last_update > 180 days
- `DECLINING_IMPRESSIONS` — impressions_last_30d < impressions_prev_30d
- `LOW_ENGAGEMENT` — engagement_rate < 40%

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write the output CSV.*

In [1]:
import pandas as pd
import numpy as np
import os

if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
else:
    df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

def minmax_norm(series, invert=False):
    """0-1 normalize; invert=True means high raw value → low score."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    norm = (series - mn) / (mx - mn)
    return 1 - norm if invert else norm

# Fill missing values with median before scoring
df['avg_position_filled'] = df['avg_position'].fillna(df['avg_position'].median())
df['ctr_filled'] = df['ctr'].fillna(df['ctr'].median())
df['days_since_last_update_filled'] = df['days_since_last_update'].fillna(df['days_since_last_update'].median())
df['engagement_rate_filled'] = df['engagement_rate'].fillna(df['engagement_rate'].median())

# Impression drop signal: last30 / prev30; cap at 2 to avoid inf
df['impression_drop_ratio'] = (
    df['impressions_last_30d'] / df['impressions_prev_30d'].clip(lower=1)
).fillna(1.0).clip(upper=2.0)

# Component scores (0 = safe, 1 = urgent)
df['s_position'] = minmax_norm(df['avg_position_filled'])          # high position# = more urgent
df['s_ctr'] = minmax_norm(df['ctr_filled'], invert=True)           # low CTR = more urgent
df['s_staleness'] = minmax_norm(df['days_since_last_update_filled']) # old = more urgent
df['s_impression_drop'] = minmax_norm(df['impression_drop_ratio'], invert=True)  # drop = urgent
df['s_engagement'] = minmax_norm(df['engagement_rate_filled'], invert=True)      # low engage = urgent

# Weighted composite score
df['baseline_score'] = (
    0.25 * df['s_position'] +
    0.20 * df['s_ctr'] +
    0.25 * df['s_staleness'] +
    0.20 * df['s_impression_drop'] +
    0.10 * df['s_engagement']
)

# Reason codes
def reason_codes(row):
    codes = []
    if row['avg_position_filled'] > 20: codes.append('HIGH_POSITION')
    if row['s_ctr'] > 0.7: codes.append('LOW_CTR')
    if row['days_since_last_update_filled'] > 180: codes.append('STALE_CONTENT')
    if row['impression_drop_ratio'] < 1.0: codes.append('DECLINING_IMPRESSIONS')
    if row['engagement_rate_filled'] < 40: codes.append('LOW_ENGAGEMENT')
    return ' | '.join(codes) if codes else 'NO_FLAG'

df['reason_codes'] = df.apply(reason_codes, axis=1)

# Sort descending by score
df_ranked = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

# Write output
os.makedirs('work/outputs', exist_ok=True)
output_cols = ['rank', 'content_id', 'client_id', 'content_type', 'baseline_score',
               'reason_codes', 'is_declining_label',
               'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate']
df_ranked[output_cols].to_csv('work/outputs/baseline_refresh_queue.csv', index=False)

print(f'Wrote {len(df_ranked):,} rows to work/outputs/baseline_refresh_queue.csv')
print(f'\nScore distribution:')
print(df_ranked['baseline_score'].describe().round(3))

Wrote 30,000 rows to work/outputs/baseline_refresh_queue.csv

Score distribution:
count    30000.000
mean         0.466
std          0.068
min          0.114
25%          0.425
50%          0.474
75%          0.513
max          0.783
Name: baseline_score, dtype: float64


## 3. Top-20 review

*Display the top 20 rows and check: do these pages look like genuine refresh candidates?*

In [2]:
# Top 20 review
top20 = df_ranked.head(20)[output_cols]
pd.set_option('display.max_colwidth', 60)
print('Top 20 pages in the refresh queue:')
print(top20.to_string(index=False))

print(f'\nTop-20 precision: {top20["is_declining_label"].mean():.1%} are truly declining')
print(f'Overall decline rate in dataset: {df["is_declining_label"].mean():.1%}')
print(f'Lift at top-20: {top20["is_declining_label"].mean() / df["is_declining_label"].mean():.2f}x')

Top 20 pages in the refresh queue:
 rank           content_id         client_id    content_type  baseline_score                                                                     reason_codes  is_declining_label  avg_position  ctr  days_since_last_update  engagement_rate
    1 content_f6fdf87348f6 client_4ec9599fc2 keyword article        0.783163 HIGH_POSITION | LOW_CTR | STALE_CONTENT | DECLINING_IMPRESSIONS | LOW_ENGAGEMENT                   1          32.5 0.00                     373              0.0
    2 content_7a888d3d99c8 client_19581e27de keyword article        0.778657 HIGH_POSITION | LOW_CTR | STALE_CONTENT | DECLINING_IMPRESSIONS | LOW_ENGAGEMENT                   1          67.6 0.00                     313              0.0
    3 content_1b4ec72dafd4 client_4ec9599fc2 keyword article        0.756471                 LOW_CTR | STALE_CONTENT | DECLINING_IMPRESSIONS | LOW_ENGAGEMENT                   1           7.0 0.00                     372              0.0
    4 content

In [3]:
# Precision@50 — the key business metric
top50 = df_ranked.head(50)
p_at_50 = top50['is_declining_label'].mean()
print(f'Precision@50: {p_at_50:.1%}')
print(f'(Of the top-50 pages in the refresh queue, {int(p_at_50*50)} are truly declining)')
print(f'\nBaseline vs random: {p_at_50:.1%} vs {df["is_declining_label"].mean():.1%}')
print(f'Lift at top-50: {p_at_50 / df["is_declining_label"].mean():.2f}x')

# Reason code distribution in top-50
print('\nReason codes in top-50:')
all_codes = ' | '.join(top50['reason_codes'].tolist())
from collections import Counter
code_list = [c.strip() for c in all_codes.split('|') if c.strip() and c.strip() != 'NO_FLAG']
print(Counter(code_list))

Precision@50: 64.0%
(Of the top-50 pages in the refresh queue, 32 are truly declining)

Baseline vs random: 64.0% vs 54.2%
Lift at top-50: 1.18x

Reason codes in top-50:
Counter({'LOW_CTR': 50, 'LOW_ENGAGEMENT': 50, 'DECLINING_IMPRESSIONS': 46, 'STALE_CONTENT': 33, 'HIGH_POSITION': 25})


## 4. What this baseline gets right and wrong

*One paragraph: honest assessment of the rule.*

**What it gets right:**
- The staleness and position signals are genuine predictors of decline — old content that is falling in rankings will continue to lose traffic without intervention.
- The reason codes make each recommendation explainable to a content manager without needing to understand the model internals.
- The weighted composite approach is transparent and auditable — a client can read the score formula and challenge individual weights.

**What it gets wrong:**
- Equal weight to staleness and position penalizes high-traffic pages that are stale but not yet declining (false positives). 
- CTR is position-dependent: a page at position 8 with 1.5% CTR is overperforming, but the rule scores it as low-CTR because the absolute number looks low compared to position-1 pages. A position-tier adjusted CTR threshold would be more honest.
- The rule is completely symmetric across clients — different industries have different baseline metrics, and a rule calibrated on the pooled data will be miscalibrated for any individual client.
- No learning from historical refresh outcomes: the rule ignores which types of content actually recovered after refresh.